# BDC Satria Data 2026 — Klasifikasi Citra Sampah
### ConvNeXt V2-Large — Eksplorasi Model Pembanding (belum predict test)

**Strategi split (tanpa holdout, sesuai kesepakatan):**
- `StratifiedShuffleSplit(n_splits=3, test_size=0.10)` → tiap fold rasio train:val = 90:10
- Metrik utama: **Macro F1-Score** (sesuai penilaian resmi BDC + kelas timpang)
- Class weighting (`balanced`) dihitung ulang tiap fold → tangani imbalance (Organic 47% / Recyclable 38% / Electronic 15%)

**Tracking misclassified → `misclassified_report.xlsx` (2 sheet):**
- **Detail** — 1 baris per kejadian salah, lengkap dengan confidence & breakdown probabilitas 3 kelas
- **Summary** — 1 baris per gambar unik, agregat lintas fold (folds_wrong, total_wrong, total_evaluated, error_rate, rata-rata probabilitas)

**Catatan model:** ConvNeXt V2 tidak tersedia di torchvision → pakai `timm`. Backbone di-freeze, hanya classifier head yang dilatih (konsisten dengan pendekatan ViT kamu).

## 1. Setup & Konfigurasi

In [1]:
from __future__ import annotations
import os
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

# timm untuk ConvNeXt V2 (tidak ada di torchvision)
try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [2]:
# === PATH DATASET ===
DATA_ROOT = Path(r"D:\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"          # belum dipakai di notebook ini
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Folder kelas: prefix angka menentukan urutan label (0=Recyclable, 1=Electronic, 2=Organic)
CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === CONFIG TRAINING ===
# Model timm untuk ConvNeXt V2-Large (pretrained ImageNet-22k -> 1k, resolusi 224)
MODEL_NAME = "convnextv2_large.fcmae_ft_in22k_in1k"
IMG_SIZE = 224
BATCH_SIZE = 8           # ConvNeXt V2-Large (~198M) jauh lebih berat dari ViT-B; turunkan dari 16 -> 8 utk VRAM 4.3GB
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4             # early stopping berdasarkan macro F1 val
N_FOLDS = 3              # sesuai permintaan: 3 fold saja
FOLD_VAL_FRAC = 0.10     # 90/10 tiap fold
SEED = 42
NUM_WORKERS = 0          # Windows/Jupyter aman di 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_NAME}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB
Model: convnextv2_large.fcmae_ft_in22k_in1k


## 2. Index Dataset dari Folder
Sama seperti notebook ViT: **tidak** load semua gambar ke memory. Cukup kumpulkan daftar `(path, label_idx)` — gambar di-load lazy per-batch lewat `Dataset`.

In [3]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

Total citra ter-index: 26527
class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64


## 3. Dataset & Transforms
Augmentasi hanya di train; val cuma resize + normalize (evaluasi jujur).

In [4]:
class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        # kembalikan juga path biar bisa dilacak saat misclassified tracking
        return img, int(row["label"]), row["path"]


def get_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

## 4. Model — ConvNeXt V2-Large (backbone freeze, head-only training)

In [5]:
def build_model(num_classes: int):
    # num_classes=0 -> timm buang classifier head, kita ganti sendiri
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)

    # Freeze semua dulu
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze hanya classifier head. Di ConvNeXt V2 timm, head ada di model.head.fc
    head = model.head
    for p in head.parameters():
        p.requires_grad = True

    return model.to(DEVICE)


# cek struktur head sekali (informational)
_tmp = build_model(NUM_CLASSES)
n_total = sum(p.numel() for p in _tmp.parameters())
n_train = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Total params: {n_total:,}")
print(f"Trainable params (head): {n_train:,}")
del _tmp
torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

d:\Downloads\BDC2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--timm--convnextv2_large.fcmae_ft_in22k_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Total params: 196,424,451
Trainable params (head): 7,683


## 5. Train & Evaluate
`evaluate` mengembalikan probabilitas softmax + path tiap sampel, supaya bisa dipakai untuk tracking misclassified.

In [6]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Cross Validation 3-Fold + Kumpulkan Misclassified

Tiap fold, setelah training selesai, model terbaik (macro F1 val tertinggi) di-evaluate ulang ke val set fold itu untuk mengambil prediksi final. Dua struktur dikumpulkan lintas fold:
- `detail_records` — tiap kejadian salah (untuk sheet **Detail**)
- `eval_counter` — berapa kali tiap gambar masuk val (untuk `total_evaluated` di sheet **Summary**)

In [7]:
sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

fold_results = []
detail_records = []                 # untuk sheet Detail
eval_counter = defaultdict(int)     # path -> berapa kali masuk val (total_evaluated)

for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
    print(f"\n{'='*64}\n---> FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    # class weight dari label training fold ini
    weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

    train_loader = DataLoader(TrashDataset(train_df, get_transforms(True)),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(TrashDataset(val_df, get_transforms(False)),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = 0.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"convnextv2_large_fold{fold}.pth"

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            marker = " [*] best"
        else:
            epochs_no_improve += 1

        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping (tidak ada peningkatan {PATIENCE} epoch).")
            break

    # Evaluate ulang pakai model terbaik fold ini utk ambil prediksi final -> tracking misclassified
    model.load_state_dict(torch.load(best_path, weights_only=True))
    _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
    print(f"  >> Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

    fold_results.append({"fold": fold, "best_macro_f1": best_f1})

    # Catat evaluasi & kesalahan
    for i, path in enumerate(paths):
        eval_counter[path] += 1
        if y_true[i] != y_pred[i]:
            p = probs[i]
            detail_records.append({
                "fold": fold,
                "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                "prob_electronic_pct": round(float(p[1]) * 100, 2),
                "prob_organic_pct": round(float(p[2]) * 100, 2),
            })

    del model, train_loader, val_loader
    torch.cuda.empty_cache()


---> FOLD 1/3 <---
  Class weights: {'Recyclable': np.float64(0.884), 'Electronic': np.float64(2.232), 'Organic': np.float64(0.704)}


  Epoch  1/15 | train_loss 0.1312 F1 0.9522 | val_loss 0.1021 F1 0.9688 [*] best
  Epoch  2/15 | train_loss 0.1057 F1 0.9648 | val_loss 0.1003 F1 0.9679
  Epoch  3/15 | train_loss 0.0948 F1 0.9677 | val_loss 0.1074 F1 0.9709 [*] best
  Epoch  4/15 | train_loss 0.0903 F1 0.9698 | val_loss 0.1055 F1 0.9705
  Epoch  5/15 | train_loss 0.0894 F1 0.9715 | val_loss 0.1094 F1 0.9692
  Epoch  6/15 | train_loss 0.0832 F1 0.9717 | val_loss 0.1053 F1 0.9732 [*] best
  Epoch  7/15 | train_loss 0.0815 F1 0.9732 | val_loss 0.1164 F1 0.9682
  Epoch  8/15 | train_loss 0.0838 F1 0.9726 | val_loss 0.1117 F1 0.9688
  Epoch  9/15 | train_loss 0.0820 F1 0.9737 | val_loss 0.1231 F1 0.9717
  Epoch 10/15 | train_loss 0.0640 F1 0.9791 | val_loss 0.1114 F1 0.9741 [*] best
  Epoch 11/15 | train_loss 0.0612 F1 0.9785 | val_loss 0.1143 F1 0.9747 [*] best
  Epoch 12/15 | train_loss 0.0606 F1 0.9787 | val_loss 0.1078 F1 0.9709
  Epoch 13/15 | train_loss 0.0605 F1 0.9789 | val_loss 0.1029 F1 0.9726
  Epoch 14/15 | tra

## 7. Ringkasan Hasil 3-Fold

In [8]:
f1s = [r["best_macro_f1"] for r in fold_results]
print("Ringkasan 3-Fold CV (macro F1):")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_macro_f1']:.4f}")
print(f"\nRata-rata macro F1: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")
print(f"\nPembanding — ViT-B/16 SWAG kamu sebelumnya: ~0.9636")

Ringkasan 3-Fold CV (macro F1):
  Fold 1: 0.9747
  Fold 2: 0.9715
  Fold 3: 0.9698

Rata-rata macro F1: 0.9720 (+/- 0.0020)

Pembanding — ViT-B/16 SWAG kamu sebelumnya: ~0.9636


## 8. Bangun Laporan Misclassified (`.xlsx`, 2 sheet)

- **Detail**: langsung dari `detail_records`.
- **Summary**: agregasi per `filepath` unik. `folds_wrong` = daftar fold tempat gambar salah; `total_evaluated` dari `eval_counter`; `error_rate = total_wrong / total_evaluated`. Diurutkan `error_rate` lalu `total_wrong` menurun → gambar paling problematik di atas.

In [10]:
detail_df = pd.DataFrame(detail_records, columns=[
    "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

# ---- Summary: agregasi per filepath unik ----
summary_rows = []
if len(detail_df) > 0:
    grouped = detail_df.groupby("filepath")
    for path, g in grouped:
        folds_wrong = sorted(g["fold"].unique().tolist())
        summary_rows.append({
            "filepath": path,
            "true_label": g["true_label"].iloc[0],   # true_label konsisten utk 1 gambar
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": eval_counter[path],
            "error_rate": round(len(g) / eval_counter[path], 3) if eval_counter[path] else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris (total kejadian salah lintas fold)")
print(f"  Summary : {len(summary_df)} gambar unik yang pernah salah")
if len(summary_df) > 0:
    print("\n5 gambar paling problematik:")
    print(summary_df.head(5).to_string(index=False))

[SAVED] D:\Downloads\BDC2026\outputs\misclassified_report.xlsx
  Detail  : 252 baris (total kejadian salah lintas fold)
  Summary : 232 gambar unik yang pernah salah

5 gambar paling problematik:
                                          filepath true_label folds_wrong  total_wrong  total_evaluated  error_rate  avg_confidence_pct  avg_true_label_confidence_pct  avg_prob_recyclable_pct  avg_prob_electronic_pct  avg_prob_organic_pct
D:\Downloads\BDC2026\train\0_Recyclable\R_1376.jpg Recyclable         1,3            2                2         1.0               99.62                           0.36                     0.36                     0.01                 99.62
D:\Downloads\BDC2026\train\0_Recyclable\R_5017.jpg Recyclable         1,3            2                2         1.0               99.21                           0.49                     0.49                     0.30                 99.21
D:\Downloads\BDC2026\train\0_Recyclable\R_8320.jpg Recyclable         1,3            2 

## Langkah selanjutnya (belum di notebook ini)
1. Bandingkan rata-rata macro F1 ConvNeXt V2-Large di atas dengan ViT-B/16 SWAG (~0.9636). Kalau kompetitif → kandidat kuat untuk stacking.
2. Buka `misclassified_report.xlsx`, cek sheet **Summary** — gambar dengan `error_rate` tinggi + `avg_confidence_pct` tinggi = prioritas cek manual (kemungkinan ambigu / salah label).
3. Kalau lanjut stacking: ganti `StratifiedShuffleSplit` → `StratifiedKFold` supaya OOF prediction bersih (tiap gambar tepat 1x di val).